# UrbanFlow - Q1 source exploration

Use a standard CPU Colab runtime in the personal Google account containing `My Drive/UrbanFlow`. This notebook verifies the archived source package and inspects source metadata and the completed run. It contains no cleaning or metric definitions; those live in the reusable package. Raw downloads go to temporary VM storage; the original source and output archives already persist in Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import hashlib
import json
import subprocess
import sys
import zipfile
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/UrbanFlow")
ORIGINAL_RUN = DRIVE / "runs/yellow-2025-q1-v1"
# This pinned archive is the exact reusable code used for the reported Q1 run.
BUNDLE = ORIGINAL_RUN / "executed-source.zip"
EXPECTED_SHA256 = "b9aeb8ea22e05f9b3d776c90dd3d4ac256fc6bb86f5da5f2554e8005d5b95aae"
if hashlib.sha256(BUNDLE.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise ValueError("Source bundle checksum mismatch")
REPO = Path("/content/urbanflow-notebook-source").resolve()
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for member in archive.infolist():
        if not (REPO / member.filename).resolve().is_relative_to(REPO):
            raise ValueError("Unsafe archive member")
    archive.extractall(REPO)
# Explicit dependencies also support the archived package before geometry/benchmark extras.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(REPO / "pipelines"),
        "pyspark==3.5.9",
        "pandas==2.2.3",
        "pyarrow>=19,<24",
        "psutil>=6,<8",
        "pyshp>=2.3,<3",
        "pyproj>=3.6,<4",
    ],
    check=True,
)
sys.path.insert(0, str(REPO / "pipelines/src"))
RAW = Path("/content/urbanflow-data/raw")
CONFIG = json.loads((REPO / "configs/yellow-2025-q1.json").read_text())
print("Verified source:", EXPECTED_SHA256)

In [ ]:
from urbanflow_etl.ingestion.download import download_dataset

source_manifest = download_dataset(CONFIG, RAW)
INPUTS = [str(RAW / item["name"]) for item in CONFIG["files"] if item["kind"] == "parquet"]
print("Validated source inputs:", INPUTS)

In [ ]:
import pyarrow.parquet as pq

for filename in INPUTS:
    parquet = pq.ParquetFile(filename)
    print(
        Path(filename).name,
        "rows:",
        parquet.metadata.num_rows,
        "row groups:",
        parquet.metadata.num_row_groups,
    )
    print(parquet.schema_arrow)
manifest = json.loads((ORIGINAL_RUN / "manifest.json").read_text())
print(
    json.dumps(
        {
            key: manifest[key]
            for key in ["run_id", "source_rows", "eligible_demand_rows", "quality", "eligibility"]
        },
        indent=2,
    )
)

A Parquet row count is not an eligible analytical count. Quality flags overlap. Read `docs/data-contracts.md` for formulas and `docs/project-report.md` for benchmark limitations. Do not infer crashes or observed road congestion from taxi records.